# Contextual Semantic Search using SPECTER

## Transformer-Based Scientific Document Embeddings

**Developer Role:** BERT Developer (Contextual Semantic Search)

---

### Overview

This notebook implements **contextual semantic representation** for research-paper search using **SPECTER** (`allenai/specter`), a transformer-based scientific document embedding model.

SPECTER is specifically designed for generating embeddings of scientific documents (titles, abstracts) and is therefore more appropriate for this project than a generic BERT model.

### What This Component Does

- Accepts a list of research-paper abstracts
- Generates **document embeddings** for the entire corpus using SPECTER
- Generates a **query embedding** for a new user query using the same model
- Returns embeddings as NumPy arrays

### What This Component Does NOT Do

- Does **not** compute cosine similarity or any distance metric
- Does **not** rank or retrieve documents
- Those responsibilities belong to a separate component

### Preprocessing Philosophy

Unlike TF-IDF or Word2Vec, transformer models rely on:
- Sentence structure and word order
- Contextual relationships between words
- Sub-word tokenization (handled internally by the model)

Therefore, we perform **only basic cleaning** (HTML tag removal, whitespace normalization) and do **not** lowercase, remove stop words, lemmatize, stem, or manually tokenize.

### Pipeline

```
Abstracts / Query
       ↓
Basic Cleaning (HTML tags, whitespace)
       ↓
SPECTER (allenai/specter)
       ↓
Embedding Vectors (NumPy arrays)
```

### Position in the Overall Comparison

| Approach | Representation | Developer |
|----------|---------------|-----------|
| TF-IDF | Lexical (exact word matching) | Developer 1 |
| Word2Vec | Static semantic (fixed word vectors) | Developer 2 |
| **SPECTER** | **Contextual scientific document embeddings** | **This notebook** |

---
## Cell 1 — Install Dependencies

Install the required libraries:
- **sentence-transformers**: Provides the `SentenceTransformer` class for loading and using pre-trained embedding models
- **torch** (PyTorch): The underlying deep-learning framework that powers the transformer model and handles GPU acceleration
- **numpy**: Used for storing and returning embedding vectors as efficient numerical arrays

The `-q` flag suppresses verbose installation output.

In [1]:
# ============================================================
# Cell 1 — Install Dependencies
# ============================================================
# sentence-transformers : loads pre-trained transformer models
#                         and generates embeddings
# torch                 : PyTorch deep-learning framework,
#                         required for GPU support
# numpy                 : stores embeddings as numerical arrays
# ============================================================

!pip install -q sentence-transformers torch numpy


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## Cell 2 — Imports

| Library | Purpose |
|---------|--------|
| `re` | Python built-in for basic text cleaning (HTML tags, whitespace) |
| `numpy` | Stores embedding matrices/vectors as NumPy arrays |
| `torch` | Checks CUDA/GPU availability, runs the transformer model |
| `SentenceTransformer` | Loads and runs the SPECTER embedding model |
| `pandas` | Loads the research-paper dataset from CSV |

In [2]:
# ============================================================
# Cell 2 — Imports
# ============================================================

import re                                       # Built-in: basic text cleaning (HTML tags, whitespace)
import numpy as np                              # Numerical arrays for storing embeddings
import torch                                    # PyTorch: GPU support and deep-learning backend
from sentence_transformers import SentenceTransformer  # Loads & runs the SPECTER model
import pandas as pd                             # Loads the dataset from CSV

print("All libraries imported successfully!")

All libraries imported successfully!


---
## Cell 3 — Check GPU Availability

Transformer models benefit significantly from GPU acceleration. This cell checks whether a CUDA-capable GPU is available:

- If **GPU is available**: The model will run on the GPU for faster embedding generation
- If **GPU is not available**: The model falls back to CPU (slower but still functional)

In [3]:
# ============================================================
# Cell 3 — Check GPU Availability
# ============================================================
# Transformer models are computationally intensive.
# A GPU significantly speeds up embedding generation.
# If no GPU is available, the model still works on CPU.
# ============================================================

print("PyTorch version:", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:            ", torch.cuda.get_device_name(0))
else:
    print("Running on CPU  (GPU not detected — this is fine, it will just be slower)")

PyTorch version: 2.11.0+cpu
CUDA available:  False
Running on CPU  (GPU not detected — this is fine, it will just be slower)


---
## Cell 4 — Load the SPECTER Model

**Model:** `allenai/specter`

SPECTER (**S**cientific **P**aper **E**mbeddings using **C**itation-informed **T**ransform**ER**s) is a transformer model specifically trained on scientific papers. It was developed by the Allen Institute for AI (AI2).

Key advantages over generic BERT:
- Trained on scientific document data (titles + abstracts)
- Uses citation signals during training to learn document similarity
- Produces embeddings that capture scientific meaning

The model is loaded onto the best available device (GPU or CPU).

In [4]:
# ============================================================
# Cell 4 — Load the SPECTER Model
# ============================================================
# SPECTER = Scientific Paper Embeddings using
#           Citation-informed TransformERs
#
# - Developed by Allen Institute for AI (allenai)
# - Specifically trained on scientific documents
# - Uses citation relationships to learn document similarity
# - More appropriate for research-paper search than generic BERT
# ============================================================

# Select the best available device: GPU if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the SPECTER model using sentence-transformers
# This downloads the model weights on first run (~400 MB)
model = SentenceTransformer(
    "allenai/specter",
    device=device
)

# Verify the model loaded correctly and check the embedding dimension
embedding_dimension = model.get_sentence_embedding_dimension()

print(f"SPECTER model loaded successfully!")
print(f"Device:              {device}")
print(f"Embedding dimension: {embedding_dimension}")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--allenai--specter. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

SPECTER model loaded successfully!
Device:              cpu
Embedding dimension: 768


C:\Users\ASUS\AppData\Local\Temp\ipykernel_10344\3904014968.py:24: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimension = model.get_sentence_embedding_dimension()


---
## Cell 5 — Define the `BERTSearcher` Class

This is the core implementation. The class:

1. **Accepts** a list of research-paper abstracts
2. **Cleans** text with only basic operations (preserving linguistic content)
3. **Generates** document embeddings for the entire corpus
4. **Generates** a query embedding for a new search query

### Preprocessing Philosophy (Critical)

Traditional NLP preprocessing (lowercasing, stop-word removal, lemmatization, stemming) **must NOT** be applied because:

- SPECTER uses its own **sub-word tokenizer** (WordPiece) that expects natural text
- The model relies on **sentence structure**, **word order**, and **context**
- Removing stop words or lowercasing destroys information the model needs

We only perform:
- HTML tag removal (`<br>`, `<p>`, etc.)
- Whitespace normalization (collapse multiple spaces into one)
- Leading/trailing whitespace stripping

In [5]:
# ============================================================
# Cell 5 — Define the BERTSearcher Class
# ============================================================

class BERTSearcher:
    """
    Contextual Semantic Search using SPECTER (allenai/specter).

    This class generates transformer-based contextual embeddings
    for research-paper abstracts and user queries using SPECTER,
    a model specifically designed for scientific documents.

    The class performs ONLY basic text cleaning and does NOT apply
    traditional NLP preprocessing (no lowercasing, no stop-word
    removal, no lemmatization, no stemming, no manual tokenization)
    because transformer models rely on the original sentence
    structure and context.

    Parameters
    ----------
    abstracts : list of str
        A list of research-paper abstracts to embed.
    specter_model : SentenceTransformer
        A pre-loaded SPECTER model instance.

    Methods
    -------
    get_corpus_embeddings()
        Returns a 2D NumPy array of shape (n_papers, embedding_dim).
    get_query_embedding(query_string)
        Returns a 1D NumPy array of shape (embedding_dim,).
    """

    def __init__(self, abstracts, specter_model):
        """
        Initialize the BERTSearcher.

        Parameters
        ----------
        abstracts : list of str
            Research-paper abstracts to generate embeddings for.
        specter_model : SentenceTransformer
            The loaded SPECTER model (allenai/specter).
        """
        # Store the raw abstracts — cleaning happens at embedding time
        self.abstracts = abstracts

        # Store the SPECTER model reference
        # Both corpus and query embeddings use the SAME model
        # so they exist in the SAME vector space
        self.model = specter_model

        print(f"BERTSearcher initialized with {len(self.abstracts)} abstracts.")
        print(f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}")

    def _clean_text(self, text):
        """
        Apply ONLY basic text cleaning.

        What this does:
        - Removes HTML tags  (e.g., <br>, <p>, </div>)
        - Collapses multiple whitespace characters into a single space
        - Strips leading and trailing whitespace

        What this does NOT do (intentionally):
        - Does NOT lowercase the text
        - Does NOT remove stop words
        - Does NOT lemmatize or stem
        - Does NOT remove punctuation
        - Does NOT manually tokenize

        The transformer's built-in tokenizer handles all tokenization.

        Parameters
        ----------
        text : str
            The raw text to clean.

        Returns
        -------
        str
            The cleaned text with linguistic content preserved.
        """
        # Remove HTML tags (e.g., <br>, <p class="...">, </div>)
        text = re.sub(r"<[^>]+>", " ", text)

        # Collapse multiple whitespace characters into a single space
        text = re.sub(r"\s+", " ", text)

        # Strip leading and trailing whitespace
        text = text.strip()

        return text

    def get_corpus_embeddings(self):
        """
        Generate embeddings for ALL research-paper abstracts.

        Pipeline:
            Raw abstracts → Basic cleaning → SPECTER → NumPy array

        Returns
        -------
        numpy.ndarray
            A 2D array of shape (n_papers, embedding_dim).
            For SPECTER, embedding_dim is typically 768.

        Example
        -------
        If there are 727 unique abstracts:
            Output shape: (727, 768)
        """
        # Step 1: Apply basic cleaning to each abstract
        # Preserves sentence structure, word order, and context
        cleaned_abstracts = [self._clean_text(abstract) for abstract in self.abstracts]

        print(f"Generating embeddings for {len(cleaned_abstracts)} abstracts...")
        print("(This may take a few minutes depending on corpus size and hardware)")

        # Step 2: Pass all cleaned abstracts through SPECTER
        # - The model's internal tokenizer handles sub-word tokenization
        # - show_progress_bar=True displays a progress bar during encoding
        # - batch_size=16 processes 16 abstracts at a time (memory-efficient)
        # - convert_to_numpy=True returns a NumPy array directly
        corpus_embeddings = self.model.encode(
            cleaned_abstracts,
            show_progress_bar=True,
            batch_size=16,
            convert_to_numpy=True
        )

        print(f"Corpus embeddings generated!")
        print(f"Shape: {corpus_embeddings.shape}")

        return corpus_embeddings

    def get_query_embedding(self, query_string):
        """
        Generate an embedding for a single user query.

        The query is processed through the EXACT SAME model
        and cleaning pipeline as the corpus abstracts, ensuring
        both exist in the same vector space.

        Pipeline:
            User query → Basic cleaning → SPECTER → NumPy array

        Parameters
        ----------
        query_string : str
            The user's search query.

        Returns
        -------
        numpy.ndarray
            A 1D array of shape (embedding_dim,).
            For SPECTER, this is typically (768,).
        """
        # Step 1: Apply the SAME basic cleaning as corpus abstracts
        cleaned_query = self._clean_text(query_string)

        # Step 2: Encode the single query through SPECTER
        # The model's internal tokenizer handles tokenization
        query_embedding = self.model.encode(
            cleaned_query,
            convert_to_numpy=True
        )

        return query_embedding

print("BERTSearcher class defined successfully!")

BERTSearcher class defined successfully!


---
## Cell 6 — Load the Dataset

Load the research-paper abstracts from `abstract_sentences.csv`.

**Important:** The CSV contains multiple rows per paper (one per annotated sentence). We use `drop_duplicates()` on the `abstract` column to get one entry per unique abstract. This guarantees the order of papers stays **exactly the same** for TF-IDF, Word2Vec, and BERT implementations.

**For Google Colab:** Upload `abstract_sentences.csv` to the Colab runtime, or mount Google Drive and adjust the path accordingly.

In [7]:
# ============================================================
# Cell 6 — Load the Dataset
# ============================================================
# The dataset "abstract_sentences.csv" is provided by the
# Scholar Inbox authors. Each row contains an abstract with
# sentence-level annotations. The same abstract appears
# multiple times, so we use drop_duplicates() to get one
# entry per unique paper.
#
# drop_duplicates() preserves the original order, ensuring
# consistency across TF-IDF, Word2Vec, and BERT implementations.
# ============================================================

# Load the dataset provided by the Scholar Inbox authors
# Make sure "abstract_sentences.csv" is in the same folder as your script
df = pd.read_csv("../abstract_sentences.csv")

# Extract the 'abstract' column and drop duplicates
# drop_duplicates() guarantees that the order of papers stays EXACTLY the same
# for TF-IDF, Word2Vec, and BERT.
unique_abstracts = df['abstract'].drop_duplicates().dropna().tolist()

print(f"Successfully loaded {len(unique_abstracts)} unique research papers!")

# ---------------------------------------------------------
# 'unique_abstracts' is now a standard Python list of strings.
# You can now pass this list into your TF-IDF, Word2Vec, or BERT code!
# ---------------------------------------------------------

# Preview the first abstract (truncated for display)
print(f"\nExample abstract (first 300 characters):")
print(f"{unique_abstracts[0][:300]}...")

Successfully loaded 727 unique research papers!

Example abstract (first 300 characters):
We present an efficient method for joint optimization of topology, materials and lighting from multi-view image observations. Unlike recent multi-view reconstruction approaches, which typically produce entangled 3D representations encoded in neural networks, we output triangle meshes with spatially-...


---
## Cell 7 — Initialize the BERTSearcher

Create an instance of `BERTSearcher` by passing:
1. The list of unique abstracts loaded from the dataset
2. The pre-loaded SPECTER model

The searcher is now ready to generate embeddings.

In [8]:
# ============================================================
# Cell 7 — Initialize the BERTSearcher
# ============================================================
# Pass the unique abstracts and the loaded SPECTER model
# to create the searcher instance.
# ============================================================

searcher = BERTSearcher(unique_abstracts, model)

BERTSearcher initialized with 727 abstracts.
Embedding dimension: 768


C:\Users\ASUS\AppData\Local\Temp\ipykernel_10344\1158504751.py:54: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


---
## Cell 8 — Generate Corpus Embeddings

Generate embeddings for **all** research-paper abstracts in the corpus.

**Expected output shape:** `(727, 768)`
- `727` = number of unique abstracts in the dataset
- `768` = SPECTER's embedding dimension (each abstract is represented as a 768-dimensional vector)

This step may take a few minutes depending on hardware (GPU vs CPU).

In [9]:
# ============================================================
# Cell 8 — Generate Corpus Embeddings
# ============================================================
# This encodes ALL abstracts through SPECTER.
# Each abstract is converted into a dense vector.
#
# Expected output: a 2D NumPy array
#   - Rows    = number of unique abstracts (727)
#   - Columns = SPECTER embedding dimension (768)
# ============================================================

corpus_embeddings = searcher.get_corpus_embeddings()

print(f"\nCorpus embeddings shape: {corpus_embeddings.shape}")

Generating embeddings for 727 abstracts...
(This may take a few minutes depending on corpus size and hardware)


Batches:   0%|          | 0/46 [00:00<?, ?it/s]

Corpus embeddings generated!
Shape: (727, 768)

Corpus embeddings shape: (727, 768)


---
## Cell 9 — Generate Query Embedding

Generate an embedding for a **new user query**.

The query goes through the **exact same** pipeline (basic cleaning → SPECTER) as the corpus abstracts, ensuring both exist in the **same vector space**.

**Expected output shape:** `(768,)`
- A single 768-dimensional vector representing the query

This vector can later be compared with the corpus embeddings to find the most relevant papers (handled by a separate component).

In [10]:
# ============================================================
# Cell 9 — Generate Query Embedding
# ============================================================
# The query is processed through the SAME model and pipeline
# as the corpus abstracts, so both embeddings exist in the
# same 768-dimensional vector space.
# ============================================================

# Example query — a natural-language search for research papers
query = "AI methods for identifying malicious network activity"

query_embedding = searcher.get_query_embedding(query)

print(f"Query: '{query}'")
print(f"Query embedding shape: {query_embedding.shape}")

Query: 'AI methods for identifying malicious network activity'
Query embedding shape: (768,)


---
## Cell 10 — Inspect Embeddings

Examine the generated embeddings to verify correctness.

This cell displays:
- Data types and shapes
- First few values of each embedding
- Summary statistics

**No similarity or ranking is performed** — that is outside the scope of this component.

In [11]:
# ============================================================
# Cell 10 — Inspect Embeddings
# ============================================================
# Verify the generated embeddings are correct.
# No similarity calculation or ranking is performed here.
# ============================================================

print("=" * 60)
print("CORPUS EMBEDDINGS")
print("=" * 60)
print(f"Type:            {type(corpus_embeddings)}")
print(f"Data type:       {corpus_embeddings.dtype}")
print(f"Shape:           {corpus_embeddings.shape}")
print(f"  → {corpus_embeddings.shape[0]} documents, each represented as a {corpus_embeddings.shape[1]}-dimensional vector")
print(f"\nFirst document embedding (first 10 values):")
print(f"  {corpus_embeddings[0][:10]}")
print(f"\nEmbedding statistics:")
print(f"  Min:  {corpus_embeddings.min():.6f}")
print(f"  Max:  {corpus_embeddings.max():.6f}")
print(f"  Mean: {corpus_embeddings.mean():.6f}")
print(f"  Std:  {corpus_embeddings.std():.6f}")

print()
print("=" * 60)
print("QUERY EMBEDDING")
print("=" * 60)
print(f"Type:            {type(query_embedding)}")
print(f"Data type:       {query_embedding.dtype}")
print(f"Shape:           {query_embedding.shape}")
print(f"  → 1 query, represented as a {query_embedding.shape[0]}-dimensional vector")
print(f"\nQuery embedding (first 10 values):")
print(f"  {query_embedding[:10]}")

print()
print("=" * 60)
print("VERIFICATION")
print("=" * 60)
print(f"Corpus and query embeddings have the same dimension: "
      f"{corpus_embeddings.shape[1] == query_embedding.shape[0]} "
      f"({corpus_embeddings.shape[1]} == {query_embedding.shape[0]})")
print(f"\n→ Both embeddings exist in the same {corpus_embeddings.shape[1]}-dimensional vector space.")
print(f"→ They can be compared using cosine similarity or other metrics")
print(f"   (handled by a separate component).")

CORPUS EMBEDDINGS
Type:            <class 'numpy.ndarray'>
Data type:       float32
Shape:           (727, 768)
  → 727 documents, each represented as a 768-dimensional vector

First document embedding (first 10 values):
  [ 0.07647364  0.4492598   0.37805587 -0.2228836   0.05148464  0.50366694
  0.10799631  0.30633122  0.16641378  0.1636046 ]

Embedding statistics:
  Min:  -9.092840
  Max:  3.589447
  Mean: -0.009352
  Std:  0.573893

QUERY EMBEDDING
Type:            <class 'numpy.ndarray'>
Data type:       float32
Shape:           (768,)
  → 1 query, represented as a 768-dimensional vector

Query embedding (first 10 values):
  [-0.44289887  0.95976377 -1.3167102   0.3581853  -0.15749298  0.91217923
  0.66916674 -0.02338852  1.2078222   1.1757766 ]

VERIFICATION
Corpus and query embeddings have the same dimension: True (768 == 768)

→ Both embeddings exist in the same 768-dimensional vector space.
→ They can be compared using cosine similarity or other metrics
   (handled by a separat

---
## Summary

### What This Notebook Produces

| Output | Variable | Shape | Description |
|--------|----------|-------|-------------|
| Corpus embeddings | `corpus_embeddings` | `(727, 768)` | One 768-dim vector per abstract |
| Query embedding | `query_embedding` | `(768,)` | One 768-dim vector for the query |

### Contribution Statement

> Implemented contextual semantic representation for research-paper search using **SPECTER** (`allenai/specter`), a transformer-based scientific document embedding model. The system preserves the original textual context rather than applying traditional preprocessing such as stop-word removal or lemmatization, and generates vector representations for both research-paper abstracts and user queries in the same embedding space.

### Position in the Overall Comparison

```
Research Paper Abstracts
          |
          +------------------+------------------+
          |                  |                  |
          v                  v                  v
       TF-IDF            Word2Vec           SPECTER
       Lexical        Static Semantic    Contextual Semantic
          |                  |                  |
          v                  v                  v
   Document Vectors    Document Vectors   Document Vectors  ← YOU ARE HERE
          |                  |                  |
          +------------------+------------------+
                             |
                             v
                    Compare Search Quality
```

In [12]:
# ============================================================
# Cell 11 — Sample Input → Output Demo
# ============================================================
# A complete walkthrough with one abstract and one query
# showing exactly what goes in and what comes out.
# ============================================================

# ── SAMPLE INPUT ─────────────────────────────────────────────
sample_abstract = (
    "We propose a novel deep learning framework for detecting "
    "cyber intrusions in network traffic. Our method combines "
    "convolutional neural networks with attention mechanisms to "
    "identify malicious patterns in packet-level data. Experiments "
    "on the CICIDS2017 benchmark demonstrate that our approach "
    "achieves 98.7% detection accuracy while maintaining low "
    "false positive rates, outperforming traditional machine "
    "learning baselines such as Random Forest and SVM."
)

sample_query = "AI methods for identifying malicious network activity"

# ── PRINT INPUT ──────────────────────────────────────────────
print("=" * 70)
print("                        INPUT")
print("=" * 70)
print()
print("SAMPLE ABSTRACT:")
print(f'  "{sample_abstract}"')
print()
print("SAMPLE QUERY:")
print(f'  "{sample_query}"')
print()

# ── GENERATE EMBEDDINGS ──────────────────────────────────────
# Create a mini searcher with just the one sample abstract
demo_searcher = BERTSearcher([sample_abstract], model)

# Generate the document embedding (1 abstract → 1 vector)
demo_doc_embedding = demo_searcher.get_corpus_embeddings()

# Generate the query embedding
demo_query_embedding = demo_searcher.get_query_embedding(sample_query)

# ── PRINT OUTPUT ─────────────────────────────────────────────
print()
print("=" * 70)
print("                        OUTPUT")
print("=" * 70)
print()
print("DOCUMENT EMBEDDING:")
print(f"  Shape : {demo_doc_embedding.shape}")
print(f"  Type  : {type(demo_doc_embedding)}")
print(f"  Dtype : {demo_doc_embedding.dtype}")
print(f"  Vector: [{demo_doc_embedding[0][0]:.6f}, {demo_doc_embedding[0][1]:.6f}, {demo_doc_embedding[0][2]:.6f}, {demo_doc_embedding[0][3]:.6f}, {demo_doc_embedding[0][4]:.6f}, ... , {demo_doc_embedding[0][-2]:.6f}, {demo_doc_embedding[0][-1]:.6f}]")
print(f"          ↑ {demo_doc_embedding.shape[1]} floating-point numbers representing this abstract")
print()
print("QUERY EMBEDDING:")
print(f"  Shape : {demo_query_embedding.shape}")
print(f"  Type  : {type(demo_query_embedding)}")
print(f"  Dtype : {demo_query_embedding.dtype}")
print(f"  Vector: [{demo_query_embedding[0]:.6f}, {demo_query_embedding[1]:.6f}, {demo_query_embedding[2]:.6f}, {demo_query_embedding[3]:.6f}, {demo_query_embedding[4]:.6f}, ... , {demo_query_embedding[-2]:.6f}, {demo_query_embedding[-1]:.6f}]")
print(f"          ↑ {demo_query_embedding.shape[0]} floating-point numbers representing this query")
print()
print("=" * 70)
print("                     WHAT THIS MEANS")
print("=" * 70)
print()
print(f"  • The abstract has been converted into a {demo_doc_embedding.shape[1]}-dimensional vector.")
print(f"  • The query   has been converted into a {demo_query_embedding.shape[0]}-dimensional vector.")
print(f"  • Both vectors live in the SAME {demo_doc_embedding.shape[1]}-dimensional space.")
print(f"  • A separate component can now compute cosine similarity")
print(f"    between them to determine how relevant the paper is to the query.")
print()
print("  Pipeline:")
print(f'    "{sample_query}"')
print(f"         ↓  SPECTER")
print(f"    [{demo_query_embedding[0]:.4f}, {demo_query_embedding[1]:.4f}, ... , {demo_query_embedding[-1]:.4f}]  ({demo_query_embedding.shape[0]}d vector)")


                        INPUT

SAMPLE ABSTRACT:
  "We propose a novel deep learning framework for detecting cyber intrusions in network traffic. Our method combines convolutional neural networks with attention mechanisms to identify malicious patterns in packet-level data. Experiments on the CICIDS2017 benchmark demonstrate that our approach achieves 98.7% detection accuracy while maintaining low false positive rates, outperforming traditional machine learning baselines such as Random Forest and SVM."

SAMPLE QUERY:
  "AI methods for identifying malicious network activity"

BERTSearcher initialized with 1 abstracts.
Embedding dimension: 768
Generating embeddings for 1 abstracts...
(This may take a few minutes depending on corpus size and hardware)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_10344\1158504751.py:54: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Corpus embeddings generated!
Shape: (1, 768)

                        OUTPUT

DOCUMENT EMBEDDING:
  Shape : (1, 768)
  Type  : <class 'numpy.ndarray'>
  Dtype : float32
  Vector: [-0.464011, 0.486656, 0.151343, 0.518930, -0.129724, ... , 0.149199, 0.664222]
          ↑ 768 floating-point numbers representing this abstract

QUERY EMBEDDING:
  Shape : (768,)
  Type  : <class 'numpy.ndarray'>
  Dtype : float32
  Vector: [-0.442899, 0.959764, -1.316710, 0.358185, -0.157493, ... , 0.823818, 1.217306]
          ↑ 768 floating-point numbers representing this query

                     WHAT THIS MEANS

  • The abstract has been converted into a 768-dimensional vector.
  • The query   has been converted into a 768-dimensional vector.
  • Both vectors live in the SAME 768-dimensional space.
  • A separate component can now compute cosine similarity
    between them to determine how relevant the paper is to the query.

  Pipeline:
    "AI methods for identifying malicious network activity"
     